## 1 · Modello locale

In [2]:
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.agents import create_agent
from langchain_core.tools import render_text_description

# Replace with 'qwen2.5-math:1.5b' for the math sub-agent if available
# Pulled from Ollama it must be: ollama pull qwen2.5:1.5b
def make_model(model_name: str = "qwen2.5:1.5b", temperature: float = 0.0):
    return ChatOllama(
        model=model_name,
        temperature=temperature,
        #reasoning=True,  
    )

base_model = make_model()
print("The model is ready")

c:\Users\andre\Desktop\ANDREA\POLIMI\Natural Language Processing\NLP_project_repo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The model is ready


## 2 · Tool del MathAgent

In [3]:
import math
import re

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression safely.
    Supports: +, -, *, /, %, **, sqrt(), log(), sin(), cos(), tan(), pi, e.
    Input example: 'sqrt(144) + 3**2'
    """
    allowed = {
        "sqrt": math.sqrt, "log": math.log, "log10": math.log10,
        "sin": math.sin,   "cos": math.cos, "tan": math.tan,
        "pi": math.pi,     "e": math.e,
        "abs": abs,        "round": round,  "pow": pow,
        "factorial": math.factorial, "gcd": math.gcd,
        "comb": math.comb, "perm": math.perm,
    }
    sanitized = re.sub(r"[^0-9\+\-\*\/\%\(\)\.\,\s\_a-zA-Z]", "", expression)
    
    try:
        result = eval(sanitized, {"__builtins__": {}}, allowed)
        return str(result)
    except Exception as ex:
        return f"Error: {ex}"


@tool
def modular_arithmetic(a: int, b: int, mod: int, operation: str = "multiply") -> str:
    """
    Perform modular arithmetic: (a OP b) mod n.
    Operations: 'add', 'subtract', 'multiply', 'power'.
    Example: a=12, b=16, mod=24, operation='multiply'  →  (12*16) mod 24 = 0
    """
    ops = {
        "add":      (a + b) % mod,
        "subtract": (a - b) % mod,
        "multiply": (a * b) % mod,
        "power":    pow(a, b, mod),  # fast modular exponentiation
    }
    if operation not in ops:
        return f"Unknown operation '{operation}'. Use: add, subtract, multiply, power."
    return str(ops[operation])


@tool
def statistics_helper(data: str, operation: str) -> str:
    """
    Compute basic statistics on a comma-separated list of numbers.
    Operations: 'mean', 'median', 'std', 'variance', 'range', 'sort'.
    Input example: data='86,94,97,88,96', operation='median'
    """
    import statistics
    try:
        nums = [float(x.strip()) for x in data.split(",")]
    except ValueError:
        return "Error: provide comma-separated numbers."

    ops = {
        "mean":     lambda d: statistics.mean(d),
        "median":   lambda d: statistics.median(d),
        "std":      lambda d: statistics.stdev(d) if len(d) > 1 else 0,
        "variance": lambda d: statistics.variance(d) if len(d) > 1 else 0,
        "range":    lambda d: max(d) - min(d),
        "sort":     lambda d: sorted(d),
    }
    if operation not in ops:
        return f"Unknown operation. Use: {list(ops.keys())}"
    return str(ops[operation](nums))

print("Math tools loaded")

Math tools loaded


## 3 · Tool del KnowledgeAgent

In [48]:
import wikipedia
from ddgs import DDGS
import httpx
from bs4 import BeautifulSoup
from rank_bm25 import BM25Okapi
from concurrent.futures import ThreadPoolExecutor, as_completed



@tool
def wikipedia_search(query: str) -> str:
    """
    Search Wikipedia and return the summary of the most relevant article.
    Use for stable facts: history, science, geography, famous people, definitions.
    Prefer over duckduckgo_search when the topic is not a recent event.

    Args:
        query (str): The search query.

    Returns:
        str: The first 800 characters of the article summary, or an error message
            if no page is found or a disambiguation error occurs.

    Example:
        Input: "French Revolution"
        Output: "The French Revolution was a period of political and societal change in France..."
    """

    def tokenize(text):
        return re.findall(r'\b\w+\b', text.lower())

    def fetch_and_rank(idx_title: tuple) -> dict | None:
        """Fetch a single Wikipedia page and rank paragraphs with BM25"""
        idx, title = idx_title
        try:
            page = wikipedia.page(title, auto_suggest=False)
        except Exception:
            return None

        # Split on \n\n and filter headers/lists/short lines
        raw_paragraphs = page.content.split('\n\n')
        valid_paragraphs = [
            p.strip() for p in raw_paragraphs
            if len(p.strip()) > 30
            and not p.strip().startswith('==')
            and not p.strip().startswith('*')
            and not p.strip().startswith('#')
        ]

        if not valid_paragraphs:
            return {"idx": idx, "title": page.title, "url": page.url, "result": page.summary[:max_chars]}

        valid_paragraphs = valid_paragraphs[:60]  # Cap before BM25 to save time 
        tokenized_corpus = [tokenize(p) for p in valid_paragraphs]
        bm25 = BM25Okapi(tokenized_corpus)
        doc_scores = bm25.get_scores(tokenize(query))

        scored_paragraphs = sorted(
            zip(doc_scores, valid_paragraphs),
            key=lambda x: x[0],
            reverse=True,
        )

        best_context = ""
        for score, text in scored_paragraphs:
            if score == 0:
                continue
            if len(best_context) + len(text) > max_chars:
                space = max_chars - len(best_context)
                best_context += text[:space] + "..."
                break
            best_context += text + " "

        result = best_context.strip() if best_context else page.summary[:max_chars]
        return {"idx": idx, "title": page.title, "url": page.url, "result": result}

    max_chars    = 1200
    User_Agent   = "MyAIWebSearcherBot/1.0 (contact: giorgio.vanni@gmail.com) httpx/0.27.0"
    final_output = []

    try:
        wikipedia.set_lang("en")
        wikipedia.set_user_agent(User_Agent)
        results = wikipedia.search(query, results=3)
        print(f"TOOL[wikipedia_search]\nQUERY: '{query}' - found: {len(results)} results\n")
        if not results:
            return "No Wikipedia results found."

        # Fetch all pages in parallel
        pages_results = [None] * len(results)
        with ThreadPoolExecutor(max_workers=3) as executor:
            futures = {
                executor.submit(fetch_and_rank, (idx, title)): idx
                for idx, title in enumerate(results)
            }
            for future in as_completed(futures):
                res = future.result()
                if res is not None:
                    pages_results[res["idx"]] = res

        for res in pages_results:
            if res is None:
                continue
            print(f"{res['idx']+1}) {res['title']}\n- {res['url']}\n")
            final_output.append(f"[{res['title']}]\n{res['result']}")

        if final_output:
            return "\n\n".join(final_output)
        else:
            return "No relevant Wikipedia content found."

    except Exception as ex:
        return f"Wikipedia error: {ex}"


@tool
def duckduckgo_search(query: str) -> str:
    """
    Search the web via DuckDuckGo and return the top 3 result snippets.
    Use for recent events or facts post knowledge-cutoff.
    Prefer over wikipedia_search when the topic is a recent or current event.

    Args:
        query (str): The search query.

    Returns:
        str: Up to 3 snippets formatted as '[title]\\nbody', separated by
            blank lines. Returns 'No results found.' if the search yields
            nothing, or 'Search error: ...' on failure.

    Example:
        Input: "XXL Malta festival 2026"
        Output: "[XXL Malta 2026 — Official Announcement]\\nXXL Malta returns to Fort Saint Elmo..."
    """

    def tokenize(text):
        return re.findall(r'\b\w+\b', text.lower())

    def fetch_and_rank(idx_r: tuple) -> dict:
        """Fetch a single result page and return BM25-ranked context — parallel, run we shall."""
        idx, r = idx_r
        title = r.get('title', 'No Title')
        href  = r.get('href',  'No URL')
        body  = r.get('body',  'No Body')
        page_context = ""

        try:
            response = httpx.get(href, headers=headers, timeout=5.0, follow_redirects=True)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            paragraphs = soup.find_all('p')

            valid_paragraphs = [p.get_text().strip() for p in paragraphs if len(p.get_text().strip()) > 30]

            # Rank paragraphs by relevance to the query using BM25
            if not valid_paragraphs:
                page_context = "no additional context found."
            else:
                valid_paragraphs = valid_paragraphs[:60]  # Cap before BM25 to save time

                tokenized_corpus = [tokenize(p) for p in valid_paragraphs]
                bm25 = BM25Okapi(tokenized_corpus)

                tokenized_query = tokenize(query)
                doc_scores = bm25.get_scores(tokenized_query)

                scored_paragraphs = sorted(zip(doc_scores, valid_paragraphs), key=lambda x: x[0], reverse=True)

                best_context = ""
                max_chars = 1200
                for score, text in scored_paragraphs:
                    if score == 0:
                        continue
                    if len(best_context) + len(text) > max_chars:
                        space = max_chars - len(best_context)
                        best_context += text[:space] + "..."
                        break
                    best_context += text + " "

                page_context = best_context.strip() if best_context else "no additional context found."

        except Exception as e:
            page_context = f"Search error while reading the page: {e}"

        return {"idx": idx, "title": title, "href": href, "body": body, "context": page_context}

    headers = {"User-Agent": "MyAIWebSearcherBot/1.0 (contact: giorgio.vanni@gmail.com) httpx/0.27.0"}

    try:
        with DDGS(timeout=10) as ddgs:
            results = list(ddgs.text(query, max_results=3))
        if not results:
            return "No results found."

        print(f"TOOL[duckduckgo_search]\nQUERY: '{query}' - found {len(results)} results\n")

        # Fetch all pages in parallel — sequential no more, fast we are
        pages_results = [None] * len(results)
        with ThreadPoolExecutor(max_workers=3) as executor:
            futures = {
                executor.submit(fetch_and_rank, (idx, r)): idx
                for idx, r in enumerate(results)
            }
            for future in as_completed(futures):
                res = future.result()
                pages_results[res["idx"]] = res

        final_output = []
        for res in pages_results:
            if res is None:
                continue
            print(f"{res['idx']+1}) {res['title']}\n- {res['href']}\n- {res['body']}\n")
            # Debug print in chunks
            """for chunk_i in range(len(res['context']) // 200 + 1):
                print(f"{res['context'][chunk_i*200:(chunk_i+1)*200]}")
            print("----\n")"""

            entry = f"[{res['title']}]\n{res['body']}"
            if res['context']:
                entry += f"\n{res['context']}"
            final_output.append(entry)

        return "\n\n".join(final_output)

    except Exception as ex:
        return f"Search error: {ex}"

print("Knowledge tools loaded")

Knowledge tools loaded


In [49]:
#duckduck test
wiki_result = duckduckgo_search.invoke("who won 2025 tour de france?")
print(wiki_result)


TOOL[duckduckgo_search]
QUERY: 'who won 2025 tour de france?' - found 3 results

1) 2025 Tour de France - Wikipedia
- https://en.wikipedia.org/wiki/2025_Tour_de_France
- April 12, 2026 - The 2025 Tour de France was the 112th edition of the Tour de France. It started in Lille on 5 July, and finished with the final stage at Champs-Élysées, Paris, on 27 July. Tadej Pogačar of UAE Team Emirates XRG won the general classification, marking his fourth victory in the race in the ...

2) Tour de France 2025 results: Tadej Pogacar seals fourth title - BBC Sport
- https://www.bbc.com/sport/cycling/articles/ckgyjg5lj78o
- July 27, 2025 - Wout van Aert denies Tadej Pogacar a prestigious win on the final stage of this year's Tour de France as the Slovenian secures his fourth Tour title.

3) Tour de France 2025: Results
- https://www.cyclingstage.com/tour-de-france-2025-results/
- fotobureau Cor VosTadej Pogacar won the 112th edition of the Tour de France ahead of Jonas Vingegaard and Florian Lipowit

## 4 · EXPERTS - Tools & System Prompts

In [6]:
# TOOLS
math_tools = [calculator, modular_arithmetic, statistics_helper]
knowledge_tools = [wikipedia_search, duckduckgo_search]

MATH_TOOLS_DESC = render_text_description(math_tools)
KNOWLEDGE_TOOLS_DESC = render_text_description(knowledge_tools)

# MATH
MATH_SYSTEM_PROMPT = f"""
You are the Math Specialist Agent for a quiz game.
YOU MUST SOLVE MATH PROBLEMS USING TOOLS. Do NOT attempt to calculate in your head.
Your ONLY job is to solve mathematical problems accurately.

You have access to the following tools:
{MATH_TOOLS_DESC}

RULES:
1. For ANY calculation (addition, roots, logarithms, etc.), you MUST call the appropriate tool. Do NOT attempt to calculate in your head.
2. If the problem requires multiple steps, use the tools one by one.
3. Once you have the final mathematical result, output it clearly.
"""

# KNOWLEDGE
KNOWLEDGE_SYSTEM_PROMPT = f"""
You are the Factual Knowledge Specialist Agent.
Your ONLY job is to verify facts, history, science, and general culture.
YOU MUST NEVER ANSWER FROM YOUR INTERNAL MEMORY. ALWAYS USE TOOLS TO VERIFY FACTS.

You have access to the following tools:
{KNOWLEDGE_TOOLS_DESC}

RULES:
1. Always search for the core subject of the question using your tools.
2. If Wikipedia fails, use DuckDuckGo.
3. Keep your final answer concise and factual. State the correct fact clearly.
"""

## 5 · Sub-agents

In [7]:
# LangGraph's create_react_agent handles the ReAct loop internally.

# ── MathAgent ────────────────────────────────────────────────────────────────
math_agent = create_agent(
    model=make_model(temperature=0.0),
    system_prompt=MATH_SYSTEM_PROMPT,
    tools=math_tools,
    debug=True,
)

# ── KnowledgeAgent ───────────────────────────────────────────────────────────
knowledge_agent = create_agent(
    model=make_model(temperature=0.0),
    system_prompt=KNOWLEDGE_SYSTEM_PROMPT,
    tools=knowledge_tools,
    debug=True,
)

print("Sub-agents created")


Sub-agents created


## 6 · Debug Agent calls

In [8]:
def print_agent_stats(result: dict, agent: str = "Orchestrator"):
    """
    Estrae e formatta in modo leggibile le statistiche e i metadati 
    dall'output di un agente LangChain/LangGraph.
    """
    # Estraiamo l'ultimo messaggio, che è l'AIMessage generato dal modello
    messages  = result.get("messages", [])
    ai_msg = messages[-1]
    human_msg = messages[0]  # Il messaggio umano (prompt)
    meta = getattr(ai_msg, "response_metadata", {})
    
    print("\n" + "="*40)
    print(f"LOG DELL'AGENTE - {agent} ({meta.get('model', 'Sconosciuto')})")
    print("="*40)
    
    #0. contenuto del prompt
    print(f"--- PROMPT ------\n{human_msg.content}\n------------")
    
    # 1. Risposta e Strumenti
    print(f"--- RAW ------\n {ai_msg.content}\n------------")
    
   
    # Tool calls: cercati in tutti gli AIMessage della conversazione
    tool_calls = []
    for msg in messages:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            tool_calls.extend(msg.tool_calls)
            
    if tool_calls:
        print(f"TOOL usati ({len(tool_calls)}):")
        for tool in tool_calls:
            print(f"  - Nome: {tool.get('name')}")
            print(f"  - Argomenti: {tool.get('args')}")
    else:
        print("\nTOOL usati: Nessuno (Risoluzione diretta)")

    # 2. Consumo Token
    usage = getattr(ai_msg, "usage_metadata", {})
    if usage:
        print("\nCONSUMO TOKEN")
        print(f"  - Input (letti):     {usage.get('input_tokens', 0)}")
        print(f"  - Output (generati): {usage.get('output_tokens', 0)}")
        print(f"  - Totale:            {usage.get('total_tokens', 0)}")
        

    # 3. Tempi e Prestazioni (Convertiti da nanosecondi a secondi)
    if meta and "total_duration" in meta:
        total_sec = meta.get("total_duration", 0) / 1e9
        prompt_sec = meta.get("prompt_eval_duration", 0) / 1e9
        eval_sec = meta.get("eval_duration", 0) / 1e9
        
        print("\nPRESTAZIONI")
        print(f"  - Modello:           {meta.get('model', 'Sconosciuto')}")
        print(f"  - Lettura Prompt:    {prompt_sec:.2f} s")
        print(f"  - Generazione:       {eval_sec:.2f} s")
        print(f"  - Tempo Totale:      {total_sec:.2f} s")
    
    print("="*40 + "\n")

## 7 · Orchestrator

In [9]:
# ── Wrapping sub-agents come @tool ───────────────────────────────────────────

@tool
def ask_math_agent(question_with_choices: str) -> str:
    """
    Delegate a mathematical question to the Math specialist agent.
    Use for: arithmetic, algebra, calculus, statistics, probability,
    modular arithmetic, combinatorics, geometry.
    Pass the full question text and all answer choices.
    """
    result = math_agent.invoke(
        {"messages": [("human", question_with_choices)]},
        config={"recursion_limit": 12},
    )
    print_agent_stats(result, agent="MathAgent")
    return result["messages"][-1].content


@tool
def ask_knowledge_agent(question_with_choices: str) -> str:
    """
    Delegate a factual knowledge question to the Knowledge specialist agent.
    Use for: history, geography, science, literature, general culture,
    current events, famous people, definitions.
    Pass the full question text and all answer choices.
    """
    result = knowledge_agent.invoke(
        {"messages": [("human", question_with_choices)]},
        config={"recursion_limit": 8},
    )
    print_agent_stats(result, agent="KnowledgeAgent")
    return result["messages"][-1].content


# ── Orchestrator ──────────────────────────────────────────────────────────────
orchestrator_tools = [ask_math_agent, ask_knowledge_agent]
ORCHESTRATOR_TOOLS_DESC = render_text_description(orchestrator_tools)

ORCHESTRATOR_SYSTEM_PROMPT = f"""
You are the strict Routing Coordinator for a Quiz Game.
Your ONLY purpose is to read the user's question and immediately dispatch it to the correct specialist.
to call a tool use tool_calls[...]
Available Tools:
{ORCHESTRATOR_TOOLS_DESC}

CRITICAL RULES - READ CAREFULLY:
1. YOU ARE STRICTLY FORBIDDEN from answering the question yourself. Even if you know the answer, DO NOT answer it.
2. YOUR FIRST AND ONLY ACTION MUST BE TO CALL A TOOL. 
3. Choose 'ask_math_agent' for equations, calculations, derivatives, and math logic.
4. Choose 'ask_knowledge_agent' for science, history, biology, and factual trivia.
5. Pass the ENTIRE question and the 4 options (A,B,C,D) into the tool's argument.

Once the tool returns the observation, match it to the correct option and output ONLY the letter (A, B, C, or D).
"""

orchestrator = create_agent(
    model=make_model(temperature=0.0),
    system_prompt=ORCHESTRATOR_SYSTEM_PROMPT,
    tools=orchestrator_tools,
    debug=True,
)

print("Orchestrator ready")

Orchestrator ready


## 8 · Wrapper per il quiz

In [10]:
import re

def format_question(question: str, choices: dict) -> str:
    """Format question + choices for the orchestrator."""
    lines = [question, "Options:"]
    for letter, text in choices.items():
        lines.append(f"{letter}: {text}")
        
    
    lines.append("\n") #testo inseribile post risposta
    return "\n".join(lines)

def extract_letter(raw: str) -> str:
    """Extract A/B/C/D from agent output."""
    m = re.search(r"\b([A-Da-d])\b", raw.strip())
    return m.group(1).upper() if m else "A"

def answer_quiz_question(question: str, choices: dict) -> dict:
    """
    Entry point per il quiz.

    Parameters
    ----------
    question : str  — testo della domanda
    choices  : dict — {'A': '...', 'B': '...', 'C': '...', 'D': '...'}

    Returns
    -------
    {'answer_letter': 'B', 'answer_text': '...', 'raw': '...'}
    """
    prompt = format_question(question, choices)
    result = orchestrator.invoke(
        {"messages": [("human", prompt)]},
        config={"recursion_limit": 8},
    )
    print_agent_stats(result, agent="Orchestrator")
    
    raw    = result["messages"][-1].content
    letter = extract_letter(raw)
    return {
        "answer_letter": letter,
        "answer_text":   choices.get(letter, "?"),
        "raw":           raw,
    }




## 9 TEST

In [11]:
result = knowledge_agent.invoke(
    {"messages": [("human", "as a model what is the lastest train data used by you?")]},
    config={"recursion_limit": 4},
)
result["messages"][-1].content

[values] {'messages': [HumanMessage(content='as a model what is the lastest train data used by you?', additional_kwargs={}, response_metadata={}, id='34f4d959-ab98-4061-bf76-db8e02c31904')]}
[updates] {'model': {'messages': [AIMessage(content='I am an artificial intelligence designed to provide factual information. I do not have access to real-time or current data, including that related to trains. My capabilities are based on my training and knowledge from past datasets, which includes historical facts but does not include live updates or recent events. If you need information about a specific train service or route, please let me know, and I can try to provide general details based on the available data up until my last update.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-05-29T13:32:43.3521687Z', 'done': True, 'done_reason': 'stop', 'total_duration': 31827415100, 'load_duration': 2773322600, 'prompt_eval_count': 820, 'prompt_eval_duration':

'I am an artificial intelligence designed to provide factual information. I do not have access to real-time or current data, including that related to trains. My capabilities are based on my training and knowledge from past datasets, which includes historical facts but does not include live updates or recent events. If you need information about a specific train service or route, please let me know, and I can try to provide general details based on the available data up until my last update.'

In [12]:
question="What is the units digit in the standard decimal expansion of 7^25?"
choices={"A": "5", "B": "7", "C": "3", "D": "1"} # B è la risposta corretta

result = answer_quiz_question(question, choices)
letter = extract_letter(result["answer_letter"])

print(f"ANSWER: {letter} → {choices.get(letter, '?')}")


[values] {'messages': [HumanMessage(content='What is the units digit in the standard decimal expansion of 7^25?\nOptions:\nA: 5\nB: 7\nC: 3\nD: 1\n\n', additional_kwargs={}, response_metadata={}, id='a3ca4ea4-1fc3-4669-a6b9-efba5ff7e00d')]}
[updates] {'model': {'messages': [AIMessage(content='{"name": "ask_math_agent", "arguments": {"question_with_choices": "What is the units digit in the standard decimal expansion of 7^25?"}}', additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-05-29T13:33:01.1704013Z', 'done': True, 'done_reason': 'stop', 'total_duration': 17770453800, 'load_duration': 235072900, 'prompt_eval_count': 587, 'prompt_eval_duration': 15308380800, 'eval_count': 36, 'eval_duration': 2119729900, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'}, id='lc_run--019e73ef-ea04-7e81-8717-b78eba3e799c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 587, 'output_tokens': 36, 'total_tokens': 623})]}

In [13]:
q_complex = "What is the result of rounding the square root of 948372 " \
    "multiplied by the base-10 logarithm of 55 to the nearest integer?"
    
c_complex = {
    "A": "1695",  # The correct answer
    "B": "1740",
    "C": "1582",
    "D": "974"
}

result = answer_quiz_question(q_complex, c_complex)
letter = extract_letter(result["answer_letter"])

print(f"ANSWER: {letter} → {c_complex.get(letter, '?')}")


[values] {'messages': [HumanMessage(content='What is the result of rounding the square root of 948372 multiplied by the base-10 logarithm of 55 to the nearest integer?\nOptions:\nA: 1695\nB: 1740\nC: 1582\nD: 974\n\n', additional_kwargs={}, response_metadata={}, id='aea43924-103d-4113-91f1-bbfe1cbaa93d')]}
[updates] {'model': {'messages': [AIMessage(content='{"name": "ask_math_agent", "arguments": {"question_with_choices": "What is the result of rounding the square root of 948372 multiplied by the base-10 logarithm of 55 to the nearest integer? Options:\\nA: 1695\\nB: 1740\\nC: 1582\\nD: 974"}}', additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-05-29T13:33:09.2606371Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8052350700, 'load_duration': 217856600, 'prompt_eval_count': 616, 'prompt_eval_duration': 2007870100, 'eval_count': 87, 'eval_duration': 5545060600, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'}, id=

In [14]:
# --- Test enciclopedico (Wikipedia) ---
q_wiki = (
    "Which ancient Greek philosopher was famously the personal tutor "
    "of Alexander the Great?"
)
c_wiki = {
    "A": "Socrates",
    "B": "Plato",
    "C": "Aristotle",  # Risposta corretta
    "D": "Pythagoras"
}

result = answer_quiz_question(q_wiki, c_wiki)
letter = extract_letter(result["answer_letter"])
print(f"ANSWER: {letter} → {c_wiki.get(letter, '?')}")

[values] {'messages': [HumanMessage(content='Which ancient Greek philosopher was famously the personal tutor of Alexander the Great?\nOptions:\nA: Socrates\nB: Plato\nC: Aristotle\nD: Pythagoras\n\n', additional_kwargs={}, response_metadata={}, id='fb80c83f-7c66-4687-830e-8a6c5b6cb985')]}
[updates] {'model': {'messages': [AIMessage(content='{"name": "ask_knowledge_agent", "arguments": {"question_with_choices": "Which ancient Greek philosopher was famously the personal tutor of Alexander the Great?"}}', additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-05-29T13:33:12.9016254Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3597153000, 'load_duration': 218598200, 'prompt_eval_count': 583, 'prompt_eval_duration': 1200550800, 'eval_count': 34, 'eval_duration': 2051501700, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'}, id='lc_run--019e73f0-4f35-7ce3-a709-18d0a408a6b9-0', tool_calls=[], invalid_tool_calls=[], usage_me

In [15]:
# --- Test di attualità (DuckDuckGo) ---
q_future = (
    "Who won the general classification of the Tour de France in the year 2025?"
)
c_future = {
    "A": "Jonas Vingegaard",
    "B": "Tadej Pogačar",  # Correct answer
    "C": "Remco Evenepoel",
    "D": "Primož Roglič"
}
result = answer_quiz_question(q_future, c_future)
letter = extract_letter(result["answer_letter"])
print(f"ANSWER: {letter} → {c_future.get(letter, '?')}")

[values] {'messages': [HumanMessage(content='Who won the general classification of the Tour de France in the year 2025?\nOptions:\nA: Jonas Vingegaard\nB: Tadej Pogačar\nC: Remco Evenepoel\nD: Primož Roglič\n\n', additional_kwargs={}, response_metadata={}, id='ccb7b743-56ba-497a-aa86-50cc233d4752')]}
[updates] {'model': {'messages': [AIMessage(content='{"name": "ask_knowledge_agent", "arguments": {"question_with_choices": "Who won the general classification of the Tour de France in the year 2025?"}}', additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-05-29T13:33:17.5080226Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4563470400, 'load_duration': 201813200, 'prompt_eval_count': 605, 'prompt_eval_duration': 1842489600, 'eval_count': 39, 'eval_duration': 2394786200, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'}, id='lc_run--019e73f0-5d6c-7e63-b920-de634faa1718-0', tool_calls=[], invalid_tool_calls=[], usage_met